In [1]:
import pandas as pd
import os
import re

# 1. Path Configuration
excel_path = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx"

if not os.path.exists(excel_path):
    raise FileNotFoundError(f"Could not find the Excel file at: {excel_path}")

# Load the file
df = pd.read_excel(excel_path)

# Clean up column names to prevent trailing/leading space KeyErrors
df.columns = df.columns.str.strip()

# Target column name variable (adjust if it's lowercase 'd' in your file)
target_col = 'Proposal Details' 

if target_col not in df.columns:
    raise KeyError(f"Could not find '{target_col}' column. Available columns are: {list(df.columns)}")

# 2. Define the Parsing Function
def extract_parivesh_details(text):
    # Fallback for empty/NaN cells
    if pd.isna(text):
        return pd.Series([None] * 5)
    
    text_str = str(text)
    
    # Using regex lookarounds to capture data between the known field labels
    clearance = re.search(r'Clearance Type:\s*(.*?)(?=\s*S/W No\.:|$)', text_str, re.IGNORECASE)
    sw_no     = re.search(r'S/W No\.\s*:\s*(.*?)(?=\s*Category:|$)', text_str, re.IGNORECASE)
    category  = re.search(r'Category\s*:\s*(.*?)(?=\s*Sector:|$)', text_str, re.IGNORECASE)
    sector    = re.search(r'Sector\s*:\s*(.*?)(?=\s*Date of Submission:|$)', text_str, re.IGNORECASE)
    date_sub  = re.search(r'Date of Submission\s*:\s*(.*?)$', text_str, re.IGNORECASE)
    
    # Extract match if found, strip trailing spaces, otherwise return None
    return pd.Series([
        clearance.group(1).strip() if clearance else None,
        sw_no.group(1).strip() if sw_no else None,
        category.group(1).strip() if category else None,
        sector.group(1).strip() if sector else None,
        date_sub.group(1).strip() if date_sub else None
    ])

# 3. Apply parsing to generate 5 new columns
print("Extracting data points from Proposal Details...")

new_cols = ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']
df[new_cols] = df[target_col].apply(extract_parivesh_details)

# 4. Save back to Excel
df.to_excel(excel_path, index=False)
print(f"Success! Extracted fields saved into columns: {new_cols}")

Extracting data points from Proposal Details...
Success! Extracted fields saved into columns: ['Clearance Type', 'S/W No.', 'Category', 'Sector', 'Date of Submission']


In [1]:
#Fill Missing values for Sector

# Define your file paths
input_path = (
    r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx"
)
output_path = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx"

# 1. Load the Excel file
df = pd.read_excel(input_path)

# Optional: Strip leading/trailing whitespaces to ensure exact matching
df["Activity Description"] = df["Activity Description"].astype(str).str.strip()

# 2. Build the lookup mapping from non-null Sector values
sector_lookup = (
    df.dropna(subset=["Sector"])
    .drop_duplicates(subset=["Activity Description"])
    .set_index("Activity Description")["Sector"]
    .to_dict()
)

# 3. Fill in missing Sector values using the Activity Description lookup
df["Sector"] = df["Sector"].fillna(df["Activity Description"].map(sector_lookup))

# 4. Save to a new Excel file
df.to_excel(output_path, index=False)

print(f"Processing complete! Saved updated file to:\n{output_path}")

Processing complete! Saved updated file to:
F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Bronze\RAW_MERGED.xlsx
